# LEPTOLIC / INT182 รอบ 3 — แนวทาง 2: Repeated nested CV

อ่าน `round3/CV_PROTOCOL_TH.md` ก่อนรัน คง NHEFS และเป้าหมายรายงานวินิจฉัยใน 8 ปี
ใช้ repeated outer 5-fold × 3 repeats และ inner 3-fold เลือกโมเดล/calibration เฉพาะ development
Primary: Brier เดิมลบใหม่ พร้อม paired bootstrap 95% CI; หากช่วงไม่เหนือ 0 ยังพิสูจน์ไม่ได้ว่าดีขึ้น
Test เดิมเคยเปิดเผยแล้ว จึงเป็น internal validation และยังไม่ยืนยันกับคนไทย

ต้องรันใน checkout ที่มีโฟลเดอร์ round3 และเข้าถึง CDC ได้
ยังไม่มีโมเดลใหม่หรือคะแนนจริงแนบมา เพราะ runtime ปัจจุบันถูก proxy บล็อก CDC 403

In [ ]:
%pip install -r round3/requirements.txt

## ตรวจหลักฐานและ protocol ก่อนฝึก
ไม่ใช้ผล Test เลือกฟีเจอร์/threshold และไม่อ้างว่า Test เดิมเป็นข้อมูลใหม่ที่ไม่เคยเห็น

In [ ]:
from pathlib import Path
from IPython.display import display, Markdown
display(Markdown(Path('round3/CV_PROTOCOL_TH.md').read_text(encoding='utf-8')))

## ฝึกด้วยข้อมูลจริง
เลือก work directory ใหม่สำหรับแต่ละ attempt
ถ้ามี ZIP จาก Colab ให้กำหนด DATA_ZIP เป็น path ของไฟล์ โปรแกรมจะตรวจและนำเข้าครบ 8 ไฟล์ก่อนฝึก โดยไม่ดาวน์โหลด CDC
หาก DATA_ZIP=None จะดาวน์โหลดจาก CDC ซึ่ง runtime ปัจจุบันยังถูกบล็อก 403
การตรวจรูปแบบ ZIP ไม่ใช่การยืนยันแหล่งกำเนิดข้อมูลอย่างอิสระ และยังต้องตรวจ codebook

In [ ]:
from round3.cv_study import run_study
WORK = Path('round3_work/nested_run_01').resolve()
DATA_ZIP = None  # เช่น Path('/content/NHEFS_data_and_layouts.zip')
results = run_study(WORK, data_zip=DATA_ZIP)

## อ่านผลและระดับ
ตรวจจำนวนคน/เหตุการณ์และช่วงความไม่แน่นอนของทุกระดับ ระดับสูงอาจมี observed rate ไม่สูงขึ้นใน Test ขนาดเล็ก ต้องรายงานตามจริง

In [ ]:
print(results['test_new'])
print(results['test_legacy_with_same_threshold_policy'])
print(results['paired_test_comparison'])
print(results['test_levels'])
display(Markdown((WORK/'report_TH.md').read_text(encoding='utf-8')))

## ทดลองกรอกข้อมูล
ตัวอย่างด้านล่างอาจถูกปฏิเสธถ้าอยู่นอก training range; ไม่เหมาะกับผู้เคยได้รับการวินิจฉัยแล้ว
ไม่ใช่เครื่องมือวินิจฉัย/เลือกการรักษา

In [ ]:
from round3.predict import predict_file
INPUT = dict(age_years=50, sex_female=0, baseline_sbp=120, baseline_dbp=75, sleep_hours=7)
answer = predict_file(WORK/'model.json', INPUT, previously_diagnosed=False)
print(answer)

## ไฟล์ผล
เมื่อรันข้อมูลจริงสำเร็จจึงมี report, metrics, model และ predictions ไม่ใช้ผล software fixtures แทนผลจริง

In [ ]:
from IPython.display import FileLink
for name in ['report_TH.md', 'metrics.json', 'model.json', 'test_predictions.csv', 'development_oof.csv']:
    display(FileLink(str(WORK/name)))